# Classification Modelling Pipeline

This notebook walks you step-by-step through building, evaluating, and preparing a **binary classification model** (XGBoost) for deployment.

**What this notebook covers:**
1. Data loading & initial inspection
2. Dropping ID / useless columns
3. Null analysis & column removal by threshold
4. Class-wise variation analysis (to detect low-information columns)
5. Categorical handling — One-Hot Encoding
6. Numerical handling — Supervised Binning then encoding
7. Saving all transformations for deployment
8. Stratified train / test split
9. XGBoost training with GridSearchCV
10. Feature importance & selection
11. Adding external / new features
12. Evaluation metrics — F1, AUC, Gini, Accuracy, Balanced Accuracy

---

## 0. Install & Import Dependencies

Run the cell below to make sure every package we need is available.

In [ ]:
# ── Install if missing ──────────────────────────────────────────────
# !pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from xgboost import XGBClassifier

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Directory where we persist artefacts for deployment
ARTEFACT_DIR = "artefacts"
os.makedirs(ARTEFACT_DIR, exist_ok=True)

print("All imports successful.")

---
## 1. Data Loading

**Instructions:**
1. Set `DATA_PATH` to the path of your CSV (or change the reader for other formats).
2. Set `TARGET_COL` to the name of your binary target column.
3. Run the cell — it will print shape, dtypes, and the first few rows so you can verify.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
DATA_PATH  = "data.csv"        # <-- change this to your file path
TARGET_COL = "target"          # <-- change this to your target column name
# ────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nTarget distribution:\n{df[TARGET_COL].value_counts(normalize=True)}")
print(f"\nColumn dtypes:\n{df.dtypes}")
df.head()

---
## 2. Drop ID / Useless Columns

**Instructions:**  
Look at the columns printed above. Identify any **ID columns**, **row-index columns**, or columns that are clearly **not useful** for prediction (e.g. names, unique identifiers, free-text keys).

Fill in the list below with those column names, then choose:
- `MODE = "drop"` → drop only the columns you list  
- `MODE = "keep"` → keep *only* the columns you list (plus the target)

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
MODE = "drop"  # "drop" or "keep"

COLUMNS_LIST = [
    # "id",
    # "customer_id",
    # "row_number",
]  # <-- fill in column names
# ────────────────────────────────────────────────────────────────────

if MODE == "drop":
    df.drop(columns=[c for c in COLUMNS_LIST if c in df.columns], inplace=True)
    print(f"Dropped {COLUMNS_LIST}")
elif MODE == "keep":
    keep = list(set(COLUMNS_LIST + [TARGET_COL]))
    df = df[[c for c in keep if c in df.columns]]
    print(f"Kept only {keep}")

print(f"Remaining shape: {df.shape}")
df.head()

---
## 3. Null Analysis & Column Removal

We will:
1. Show the **percentage of nulls** per column.
2. Drop every column whose null % exceeds the threshold you set.

**Instructions:**  
Set `NULL_THRESHOLD_PCT` — any column with a higher null percentage will be removed.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
NULL_THRESHOLD_PCT = 40  # drop columns with more than this % nulls
# ────────────────────────────────────────────────────────────────────

null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct_df = null_pct.reset_index()
null_pct_df.columns = ["column", "null_pct"]

# Visualise
cols_with_nulls = null_pct_df[null_pct_df["null_pct"] > 0]
if len(cols_with_nulls) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(cols_with_nulls) * 0.4)))
    sns.barplot(data=cols_with_nulls, x="null_pct", y="column", ax=ax,
                palette="Reds_r")
    ax.axvline(NULL_THRESHOLD_PCT, color="black", ls="--", label=f"Threshold {NULL_THRESHOLD_PCT}%")
    ax.set_xlabel("Null %")
    ax.set_title("Null Percentage by Column")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No columns have null values.")

# Drop
cols_to_drop = null_pct[null_pct > NULL_THRESHOLD_PCT].index.tolist()
if TARGET_COL in cols_to_drop:
    cols_to_drop.remove(TARGET_COL)
df.drop(columns=cols_to_drop, inplace=True)
print(f"\nDropped {len(cols_to_drop)} columns above {NULL_THRESHOLD_PCT}% nulls: {cols_to_drop}")
print(f"Remaining shape: {df.shape}")

---
## 4. Class-wise Variation Analysis

For each feature we compare its distribution across the two target classes.  
Columns that look **almost identical** in both classes carry little predictive power and can be dropped.

**How we measure it:**
- **Numerical columns** → Kolmogorov-Smirnov test (p-value). High p-value = distributions are the same.
- **Categorical columns** → Cramér's V. Low V = no association with target.

**Instructions:**  
Set `VARIATION_THRESHOLD` — columns below this threshold will be flagged.  
Review the flagged columns and decide which to drop.

In [ ]:
from scipy.stats import ks_2samp, chi2_contingency

def cramers_v(col, target):
    """Cramér's V between a categorical column and binary target."""
    ct = pd.crosstab(col, target)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0

# ── USER INPUT ──────────────────────────────────────────────────────
VARIATION_THRESHOLD = 0.02  # columns with variation score below this get flagged
# ────────────────────────────────────────────────────────────────────

feature_cols = [c for c in df.columns if c != TARGET_COL]
variation_scores = {}

for col in feature_cols:
    if df[col].nunique() < 2:
        variation_scores[col] = 0.0
        continue
    if df[col].dtype in ["object", "category"] or df[col].nunique() <= 10:
        variation_scores[col] = cramers_v(df[col].fillna("__NULL__"), df[TARGET_COL])
    else:
        grp0 = df.loc[df[TARGET_COL] == df[TARGET_COL].unique()[0], col].dropna()
        grp1 = df.loc[df[TARGET_COL] == df[TARGET_COL].unique()[1], col].dropna()
        if len(grp0) == 0 or len(grp1) == 0:
            variation_scores[col] = 0.0
        else:
            ks_stat, _ = ks_2samp(grp0, grp1)
            variation_scores[col] = ks_stat  # higher = more different

var_df = (
    pd.DataFrame.from_dict(variation_scores, orient="index", columns=["variation_score"])
    .sort_values("variation_score", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(4, len(var_df) * 0.35)))
colors = ["#d62728" if v < VARIATION_THRESHOLD else "#2ca02c" for v in var_df["variation_score"]]
sns.barplot(x=var_df["variation_score"], y=var_df.index, palette=colors, ax=ax)
ax.axvline(VARIATION_THRESHOLD, ls="--", color="black", label=f"Threshold={VARIATION_THRESHOLD}")
ax.set_title("Class-wise Variation Score per Feature")
ax.set_xlabel("Variation Score (higher = more discriminative)")
ax.legend()
plt.tight_layout()
plt.show()

flagged = var_df[var_df["variation_score"] < VARIATION_THRESHOLD].index.tolist()
print(f"\nFlagged columns (score < {VARIATION_THRESHOLD}): {flagged}")

**Instructions:**  
Review the flagged columns above. Add any you want to drop to the list below.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
DROP_LOW_VARIATION = [
    # "col_a",
    # "col_b",
]  # <-- paste flagged column names you want to remove
# ────────────────────────────────────────────────────────────────────

df.drop(columns=[c for c in DROP_LOW_VARIATION if c in df.columns], inplace=True)
print(f"Dropped {len(DROP_LOW_VARIATION)} low-variation columns.")
print(f"Remaining shape: {df.shape}")

---
## 5. Identify Categorical vs Numerical Columns

The notebook auto-detects column types.  
Review the lists and move columns between them if the auto-detection was wrong.

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COL]

cat_cols = [c for c in feature_cols
            if df[c].dtype in ["object", "category"] or df[c].nunique() <= 10]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns  ({len(num_cols)}): {num_cols}")

**Instructions:**  
If any column was misclassified, move it manually in the cell below.

In [ ]:
# ── USER INPUT (optional) ──────────────────────────────────────────
# Move columns between lists if the auto-detection was wrong:
# cat_cols.append("some_numeric_col_that_is_actually_categorical")
# num_cols.remove("some_numeric_col_that_is_actually_categorical")
# ────────────────────────────────────────────────────────────────────

print(f"Final categorical: {cat_cols}")
print(f"Final numerical:   {num_cols}")

---
## 6. Numerical Columns — Binning

For each numerical column we will:
1. Show its distribution and the **binning conditions** before applying.
2. Bin it into discrete intervals.
3. Save the bin edges so we can apply the same transform at deployment.

**Instructions:**  
- `N_BINS` — default number of bins.  
- `BINNING_STRATEGY` — `"quantile"` (equal-frequency), `"uniform"` (equal-width), or `"kmeans"`.  
- You can also define **custom bin edges** per column in `CUSTOM_BINS` if you need specific cut-points.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
N_BINS            = 5
BINNING_STRATEGY  = "quantile"  # "quantile", "uniform", or "kmeans"

# Optional: specify custom bin edges for specific columns.
# Example: CUSTOM_BINS = {"age": [0, 18, 35, 55, 100]}
CUSTOM_BINS = {}
# ────────────────────────────────────────────────────────────────────

bin_edges_store = {}  # will be saved for deployment

for col in num_cols:
    series = df[col].dropna()
    print(f"\n{'='*60}")
    print(f"Column: {col}")
    print(f"  min={series.min():.4f}  max={series.max():.4f}  "
          f"mean={series.mean():.4f}  median={series.median():.4f}  "
          f"std={series.std():.4f}  nulls={df[col].isnull().sum()}")

    if col in CUSTOM_BINS:
        edges = CUSTOM_BINS[col]
        df[col + "_bin"] = pd.cut(df[col], bins=edges, labels=False,
                                   include_lowest=True)
        bin_edges_store[col] = {"type": "custom", "edges": edges}
        print(f"  → Custom bins applied: {edges}")
    else:
        # Show planned quantile/uniform edges before applying
        if BINNING_STRATEGY == "quantile":
            quantiles = np.linspace(0, 1, N_BINS + 1)
            edges = list(series.quantile(quantiles).values)
            print(f"  → Quantile bin edges: {[round(e, 4) for e in edges]}")
        elif BINNING_STRATEGY == "uniform":
            edges = list(np.linspace(series.min(), series.max(), N_BINS + 1))
            print(f"  → Uniform bin edges:  {[round(e, 4) for e in edges]}")
        else:
            edges = None
            print(f"  → KMeans binning with {N_BINS} bins (edges computed by KMeans)")

        binner = KBinsDiscretizer(
            n_bins=N_BINS, encode="ordinal", strategy=BINNING_STRATEGY,
            subsample=None,
        )
        valid_mask = df[col].notna()
        df.loc[valid_mask, col + "_bin"] = binner.fit_transform(
            df.loc[valid_mask, [col]]
        ).ravel()

        actual_edges = binner.bin_edges_[0].tolist()
        bin_edges_store[col] = {
            "type": BINNING_STRATEGY,
            "edges": actual_edges,
            "n_bins": N_BINS,
        }
        print(f"  → Actual bin edges saved: {[round(e, 4) for e in actual_edges]}")

    print(f"  → Value counts after binning:")
    print(df[col + "_bin"].value_counts().sort_index())

# Drop original numerical columns — we keep the binned versions
df.drop(columns=num_cols, inplace=True)
print(f"\nOriginal numerical columns dropped. Shape: {df.shape}")

---
## 7. Categorical Columns — One-Hot Encoding

Each categorical column (including the newly binned numerical columns) will be one-hot encoded.  
The encoder is saved so the same mapping can be reused at deployment.

**Instructions:**  
- `DROP_FIRST` — set to `True` to drop one dummy per feature (avoids multicollinearity for linear models; for tree models you can leave `False`).  
- `MAX_CATEGORIES` — categories with fewer than this many occurrences are grouped into `"__rare__"`.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
DROP_FIRST      = False   # drop first dummy column per feature?
MAX_CATEGORIES  = None    # set an int to group rare categories, or None to skip
# ────────────────────────────────────────────────────────────────────

# Identify all columns to encode (original cat + binned)
encode_cols = [c for c in df.columns if c != TARGET_COL]

# Group rare categories if requested
rare_mappings = {}
if MAX_CATEGORIES is not None:
    for col in encode_cols:
        counts = df[col].value_counts()
        rare_cats = counts[counts < MAX_CATEGORIES].index.tolist()
        if rare_cats:
            df[col] = df[col].apply(lambda x: "__rare__" if x in rare_cats else x)
            rare_mappings[col] = rare_cats
            print(f"{col}: grouped {len(rare_cats)} rare categories into '__rare__'")

# Fill remaining nulls before encoding
for col in encode_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna("__NULL__")

# Fit OneHotEncoder
df[encode_cols] = df[encode_cols].astype(str)
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore",
                    drop="first" if DROP_FIRST else None)
encoded = ohe.fit_transform(df[encode_cols])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(encode_cols),
                          index=df.index)

# Rebuild dataframe
df = pd.concat([encoded_df, df[[TARGET_COL]]], axis=1)
print(f"\nShape after one-hot encoding: {df.shape}")
df.head()

---
## 8. Save All Transformation Artefacts

Everything needed to reproduce these transformations on new data is saved here:  
- Bin edges (JSON)  
- OneHotEncoder (joblib)  
- Rare-category mappings (JSON)  
- Column lists (JSON)  

At deployment, load these artefacts and apply the same pipeline.

In [ ]:
# Save bin edges
with open(os.path.join(ARTEFACT_DIR, "bin_edges.json"), "w") as f:
    json.dump(bin_edges_store, f, indent=2)

# Save OHE
joblib.dump(ohe, os.path.join(ARTEFACT_DIR, "ohe_encoder.joblib"))

# Save rare mappings
with open(os.path.join(ARTEFACT_DIR, "rare_mappings.json"), "w") as f:
    json.dump(rare_mappings, f, indent=2)

# Save column lists
col_meta = {
    "original_cat_cols": cat_cols,
    "original_num_cols": num_cols,
    "encode_cols": encode_cols,
    "final_feature_cols": [c for c in df.columns if c != TARGET_COL],
    "target_col": TARGET_COL,
}
with open(os.path.join(ARTEFACT_DIR, "column_metadata.json"), "w") as f:
    json.dump(col_meta, f, indent=2)

print("All transformation artefacts saved to:", ARTEFACT_DIR)
print("Files:", os.listdir(ARTEFACT_DIR))

---
## 9. Stratified Train / Test Split

We use **stratified splitting** so both train and test sets preserve the original class ratio.

**Instructions:**  
- `TEST_SIZE` — fraction of data reserved for testing.  
- `RANDOM_STATE` — seed for reproducibility.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
TEST_SIZE    = 0.2
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────────────────

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train target dist:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest target dist:\n{y_test.value_counts(normalize=True)}")

---
## 10. XGBoost — GridSearchCV for Hyperparameters

We use `GridSearchCV` with **stratified k-fold** cross-validation to find the best XGBoost parameters.

**Instructions:**  
- Edit `param_grid` to change the search space.  
- Set `CV_FOLDS` and `SCORING` metric.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
CV_FOLDS = 5
SCORING  = "roc_auc"  # scoring metric for grid search

param_grid = {
    "n_estimators":     [100, 200, 300],
    "max_depth":        [3, 5, 7],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}
# ────────────────────────────────────────────────────────────────────

# Calculate scale_pos_weight for imbalanced data
neg, pos = np.bincount(y_train.astype(int))
scale_pos_weight = neg / pos if pos > 0 else 1
print(f"scale_pos_weight = {scale_pos_weight:.2f} (neg={neg}, pos={pos})")

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    verbosity=0,
)

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring=SCORING,
    cv=skf,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print("\nStarting GridSearchCV ... (this may take a while)")
grid_search.fit(X_train, y_train)

print(f"\nBest {SCORING}: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

In [ ]:
# Store the best estimator
best_model = grid_search.best_estimator_

# Show top-10 parameter combos
cv_results = pd.DataFrame(grid_search.cv_results_)
top10 = cv_results.nsmallest(10, "rank_test_score")[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
]
top10

---
## 11. Feature Importance & Selection

We examine which features the model considers most important.  
This helps you understand the model and decide whether to remove low-importance features.

Three views are provided:
1. **XGBoost built-in importance** (gain-based)
2. **Permutation importance** on the test set
3. **Summary table** with optional feature removal

In [ ]:
from sklearn.inspection import permutation_importance

# ── 1. Built-in importance ──────────────────────────────────────────
imp = pd.Series(best_model.feature_importances_, index=X_train.columns)
imp_sorted = imp.sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(imp_sorted) * 0.25)))

# Left: XGBoost gain importance
top_n = min(30, len(imp_sorted))
sns.barplot(x=imp_sorted.values[:top_n], y=imp_sorted.index[:top_n],
            palette="viridis", ax=axes[0])
axes[0].set_title("XGBoost Feature Importance (Gain)")
axes[0].set_xlabel("Importance")

# ── 2. Permutation importance ──────────────────────────────────────
perm_result = permutation_importance(
    best_model, X_test, y_test, n_repeats=10,
    random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1,
)
perm_imp = pd.Series(perm_result.importances_mean, index=X_test.columns)
perm_sorted = perm_imp.sort_values(ascending=False)

sns.barplot(x=perm_sorted.values[:top_n], y=perm_sorted.index[:top_n],
            palette="magma", ax=axes[1])
axes[1].set_title("Permutation Importance (AUC drop)")
axes[1].set_xlabel("Mean AUC decrease")

plt.tight_layout()
plt.show()

# ── 3. Summary table ───────────────────────────────────────────────
summary = pd.DataFrame({
    "xgb_importance": imp,
    "perm_importance": perm_imp,
}).sort_values("perm_importance", ascending=False)
print("\nFull feature importance summary:")
summary

**Instructions:**  
Review the importance scores above.  
If you want to **drop low-importance features** and retrain, list them below.

In [ ]:
# ── USER INPUT (optional) ──────────────────────────────────────────
DROP_FEATURES = [
    # "feature_x_bin_2.0",
]  # <-- add feature names to drop and retrain
# ────────────────────────────────────────────────────────────────────

if DROP_FEATURES:
    X_train = X_train.drop(columns=DROP_FEATURES, errors="ignore")
    X_test  = X_test.drop(columns=DROP_FEATURES, errors="ignore")

    print(f"Dropped {len(DROP_FEATURES)} features. Retraining ...")
    best_model.fit(X_train, y_train)
    print("Retrained.")
else:
    print("No features dropped — using full feature set.")

---
## 12. Adding New / External Features

Sometimes external data providers can supply features that improve performance.  
Below you can **merge additional feature files** with the existing data and evaluate the impact.

**Instructions:**  
1. Set `EXTERNAL_DATA_PATH` to the CSV with new features.  
2. Set `MERGE_KEY` to the column to join on (must exist in both datasets).  
3. Run the cell — it will merge, retrain, and print before/after AUC for comparison.

If you don't have external data, skip this section.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
EXTERNAL_DATA_PATH = None   # e.g. "external_features.csv"  — set to None to skip
MERGE_KEY          = None   # e.g. "customer_id"
# ────────────────────────────────────────────────────────────────────

if EXTERNAL_DATA_PATH is not None and MERGE_KEY is not None:
    ext_df = pd.read_csv(EXTERNAL_DATA_PATH)
    print(f"External data shape: {ext_df.shape}")
    print(f"External columns: {ext_df.columns.tolist()}")

    # Evaluate BEFORE adding new features
    y_prob_before = best_model.predict_proba(X_test)[:, 1]
    auc_before = roc_auc_score(y_test, y_prob_before)

    # Merge — this assumes your train/test still have the merge key as index or column
    # Adjust as needed.
    new_features = ext_df.drop(columns=[MERGE_KEY], errors="ignore")
    # One-hot encode / bin new features the same way as before, or add them raw
    # For simplicity we add them raw here — adjust processing as needed.
    X_train_ext = X_train.join(new_features, how="left")
    X_test_ext  = X_test.join(new_features, how="left")
    X_train_ext.fillna(0, inplace=True)
    X_test_ext.fillna(0, inplace=True)

    # Retrain
    model_ext = best_model.__class__(**best_model.get_params())
    model_ext.fit(X_train_ext, y_train)
    y_prob_after = model_ext.predict_proba(X_test_ext)[:, 1]
    auc_after = roc_auc_score(y_test, y_prob_after)

    print(f"\nAUC BEFORE external features: {auc_before:.4f}")
    print(f"AUC AFTER  external features: {auc_after:.4f}")
    print(f"Delta: {auc_after - auc_before:+.4f}")

    if auc_after > auc_before:
        print("\n→ External features IMPROVED the model. Keeping them.")
        X_train, X_test = X_train_ext, X_test_ext
        best_model = model_ext
    else:
        print("\n→ External features did NOT improve the model. Discarding them.")
else:
    print("No external data provided — skipping this step.")

---
## 13. Evaluation Metrics

We compute and visualise the following metrics on the **test set**:

| Metric | What it tells you |
|--------|-------------------|
| **Accuracy** | Overall correctness |
| **Balanced Accuracy** | Average recall per class — good for imbalanced data |
| **F1 Score** | Harmonic mean of precision and recall |
| **AUC (ROC)** | Rank-ordering ability |
| **Gini** | `2 × AUC − 1` — common in credit scoring |

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

acc      = accuracy_score(y_test, y_pred)
bal_acc  = balanced_accuracy_score(y_test, y_pred)
f1       = f1_score(y_test, y_pred)
auc      = roc_auc_score(y_test, y_prob)
gini     = 2 * auc - 1

metrics = {
    "Accuracy":          acc,
    "Balanced Accuracy": bal_acc,
    "F1 Score":          f1,
    "AUC (ROC)":         auc,
    "Gini":              gini,
}

print("="*50)
print("         TEST SET METRICS")
print("="*50)
for name, val in metrics.items():
    bar = "|" + "█" * int(val * 40) + " " * (40 - int(val * 40)) + "|"
    print(f"  {name:<20s}  {val:.4f}  {bar}")
print("="*50)
print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# ── Beautiful metric visualisations ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# ── 1. ROC Curve ───────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0, 0].plot(fpr, tpr, lw=2, color="#1f77b4",
                label=f"AUC = {auc:.4f}")
axes[0, 0].plot([0, 1], [0, 1], "--", color="grey")
axes[0, 0].fill_between(fpr, tpr, alpha=0.15, color="#1f77b4")
axes[0, 0].set_xlabel("False Positive Rate")
axes[0, 0].set_ylabel("True Positive Rate")
axes[0, 0].set_title("ROC Curve")
axes[0, 0].legend(loc="lower right", fontsize=12)

# ── 2. Precision-Recall Curve ──────────────────────────────────────
prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[0, 1].plot(rec, prec, lw=2, color="#ff7f0e")
axes[0, 1].fill_between(rec, prec, alpha=0.15, color="#ff7f0e")
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].set_title("Precision-Recall Curve")

# ── 3. Confusion Matrix ────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1, 0],
            xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
axes[1, 0].set_xlabel("Predicted")
axes[1, 0].set_ylabel("Actual")
axes[1, 0].set_title("Confusion Matrix")

# ── 4. Metrics Bar Chart ───────────────────────────────────────────
colors_bar = ["#2ca02c", "#17becf", "#ff7f0e", "#1f77b4", "#9467bd"]
bars = axes[1, 1].bar(metrics.keys(), metrics.values(), color=colors_bar,
                       edgecolor="black", linewidth=0.8)
for bar, val in zip(bars, metrics.values()):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.4f}", ha="center", fontsize=11, fontweight="bold")
axes[1, 1].set_ylim(0, 1.1)
axes[1, 1].set_title("Metric Summary")
axes[1, 1].tick_params(axis="x", rotation=25)

plt.suptitle("Model Evaluation Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 14. Save Final Model & Artefacts for Deployment

We save:
- The trained XGBoost model
- The final feature list
- A summary of metrics

Together with the transformation artefacts saved in Section 8, this gives you everything needed to deploy.

In [ ]:
# Save model
joblib.dump(best_model, os.path.join(ARTEFACT_DIR, "xgb_model.joblib"))

# Save final feature list
with open(os.path.join(ARTEFACT_DIR, "final_features.json"), "w") as f:
    json.dump(X_train.columns.tolist(), f, indent=2)

# Save metrics
with open(os.path.join(ARTEFACT_DIR, "test_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Save best hyperparameters
with open(os.path.join(ARTEFACT_DIR, "best_params.json"), "w") as f:
    json.dump(grid_search.best_params_, f, indent=2)

print("All deployment artefacts saved.")
print("Contents of artefact directory:")
for fname in sorted(os.listdir(ARTEFACT_DIR)):
    size = os.path.getsize(os.path.join(ARTEFACT_DIR, fname))
    print(f"  {fname:<30s}  {size:>8,} bytes")

---
## 15. How to Use These Artefacts at Deployment

Below is a reference function that loads all artefacts and scores a new raw dataframe.

```python
import joblib, json, pandas as pd, numpy as np

def score_new_data(raw_df, artefact_dir="artefacts"):
    # 1. Load artefacts
    model      = joblib.load(f"{artefact_dir}/xgb_model.joblib")
    ohe        = joblib.load(f"{artefact_dir}/ohe_encoder.joblib")
    bin_edges  = json.load(open(f"{artefact_dir}/bin_edges.json"))
    col_meta   = json.load(open(f"{artefact_dir}/column_metadata.json"))
    rare_map   = json.load(open(f"{artefact_dir}/rare_mappings.json"))
    features   = json.load(open(f"{artefact_dir}/final_features.json"))

    # 2. Bin numerical columns
    for col, info in bin_edges.items():
        edges = info["edges"]
        raw_df[col + "_bin"] = pd.cut(
            raw_df[col], bins=edges, labels=False, include_lowest=True
        )
    raw_df.drop(columns=col_meta["original_num_cols"], inplace=True, errors="ignore")

    # 3. Handle rare categories
    for col, rares in rare_map.items():
        if col in raw_df.columns:
            raw_df[col] = raw_df[col].apply(
                lambda x: "__rare__" if x in rares else x
            )

    # 4. Fill nulls & encode
    encode_cols = col_meta["encode_cols"]
    for col in encode_cols:
        if col in raw_df.columns and raw_df[col].isnull().any():
            raw_df[col] = raw_df[col].fillna("__NULL__")
    raw_df[encode_cols] = raw_df[encode_cols].astype(str)
    encoded = ohe.transform(raw_df[encode_cols])
    enc_df  = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(encode_cols),
                           index=raw_df.index)

    # 5. Align to training features and predict
    for col in features:
        if col not in enc_df.columns:
            enc_df[col] = 0
    enc_df = enc_df[features]

    probabilities = model.predict_proba(enc_df)[:, 1]
    return probabilities
```

---

**You're done!**  
Go back to any section to tweak thresholds, change binning strategies, or add features — then re-run from that point onward.